# Module 12: Differential Equations for AI

Differential equations describe systems that change continuously over time or space. Recently, they have become integral to AI through **Neural Ordinary Differential Equations (Neural ODEs)**—which treat residual neural networks as continuous dynamical systems—and **Score-Based Generative Models (Diffusion Models)**—which use stochastic differential equations to model data generation. This module covers the mathematical foundations of ODEs, PDEs, SDEs, and their AI applications.

## Contents
1. Ordinary Differential Equations (ODEs) & Dynamical Systems
2. Numerical Integration (Euler, Runge-Kutta, Adaptive Step Sizes)
3. Stability Analysis & Phase Portraits
4. Neural ODEs & The Adjoint Sensitivity Method
5. Stochastic Differential Equations (SDEs) & Diffusion Models
6. Partial Differential Equations (PDEs) & Physics-Informed Neural Networks (PINNs)

## 1. Ordinary Differential Equations & Dynamical Systems

An **Ordinary Differential Equation (ODE)** relates a function $y(t)$ to its derivatives with respect to a single independent variable $t$:
$$\frac{dy}{dt} = f(y(t), t)$$
Given an initial condition $y(0) = y_0$, this forms an **Initial Value Problem (IVP)**.

In a **linear system of ODEs**:
$$\frac{d\mathbf{y}}{dt} = A\mathbf{y}$$
the solution is given by the matrix exponential:
$$\mathbf{y}(t) = e^{At} \mathbf{y}_0$$

Let's solve a simple first-order ODE using SymPy.

In [ ]:
import sympy as sp

# Solve dy/dt = -2y with y(0) = 5
t = sp.Symbol('t')
y = sp.Function('y')
eq = sp.Eq(y(t).diff(t), -2 * y(t))

sol = sp.dsolve(eq, ics={y(0): 5})
print("Analytical Solution:")
sp.pprint(sol)

## 2. Numerical Integration

When $f(y, t)$ is non-linear, we cannot find analytical solutions. We integrate numerically:

- **Euler's Method** (First-order):
  $$y_{n+1} = y_n + h f(y_n, t_n)$$
- **Runge-Kutta 4th Order (RK4)** (Fourth-order):
  $$k_1 = f(y_n, t_n)$$
  $$k_2 = f(y_n + \frac{h}{2} k_1, t_n + \frac{h}{2})$$
  $$k_3 = f(y_n + \frac{h}{2} k_2, t_n + \frac{h}{2})$$
  $$k_4 = f(y_n + h k_3, t_n + h)$$
  $$y_{n+1} = y_n + \frac{h}{6}(k_1 + 2k_2 + 2k_3 + k_4)$$
- **Adaptive Solvers** (e.g. Dormand-Prince / RK45): Dynamic step size selection based on local error estimation.

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# Lorenz system (chaotic system)
def lorenz(t, state, sigma=10, rho=28, beta=8/3):
    x, y, z = state
    dxdt = sigma * (y - x)
    dydt = x * (rho - z) - y
    dzdt = x * y - beta * z
    return [dxdt, dydt, dzdt]

sol = solve_ivp(lorenz, (0, 40), [1.0, 1.0, 1.0], t_eval=np.linspace(0, 40, 5000))

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(projection='3d')
ax.plot(sol.y[0], sol.y[1], sol.y[2], lw=0.5)
ax.set_title("Lorenz Attractor")
plt.show()

## 3. Stability Analysis & Phase Portraits

To analyze a system $\frac{d\mathbf{y}}{dt} = f(\mathbf{y})$ without solving it, we look at **equilibrium points** where $f(\mathbf{y}^*) = 0$.

We linearize around $\mathbf{y}^*$ using the Jacobian $J = \nabla f(\mathbf{y}^*)$:
$$\frac{d\Delta\mathbf{y}}{dt} \approx J \Delta\mathbf{y}$$

The eigenvalues of $J$ dictate the local behavior:
- **All eigenvalues have negative real part**: Stable equilibrium (sink/node/spiral).
- **Any eigenvalue has positive real part**: Unstable equilibrium (source/saddle).
- **Pure imaginary eigenvalues**: Neutral/oscillatory equilibrium (center).

## 4. Neural ODEs & The Adjoint Sensitivity Method

A standard Residual Network (ResNet) has updates:
$$\mathbf{h}_{t+1} = \mathbf{h}_t + f(\mathbf{h}_t, \theta_t)$$
If we take the limit of infinitesimal step sizes, we get a continuous ODE:
$$\frac{d\mathbf{h}(t)}{dt} = f(\mathbf{h}(t), t, \theta)$$

### Adjoint Sensitivity Method
Computing gradients directly through a numerical solver requires caching all intermediate steps, which is memory-intensive ($O(T)$). Instead, Neural ODEs use the **adjoint sensitivity method**, solving an auxiliary ODE backward in time to get gradients with $O(1)$ memory cost.

Let $L$ be the scalar loss. We define the adjoint state $\mathbf{a}(t) = \frac{\partial L}{\partial \mathbf{h}(t)}$, which satisfies:
$$\frac{d\mathbf{a}(t)}{dt} = -\mathbf{a}(t)^T \frac{\partial f(\mathbf{h}(t), t, \theta)}{\partial \mathbf{h}}$$

## 5. Stochastic Differential Equations & Diffusion Models

A **Stochastic Differential Equation (SDE)** incorporates randomness/noise:
$$d\mathbf{x} = \mathbf{f}(\mathbf{x}, t) dt + g(t) d\mathbf{w}$$
where $\mathbf{f}(\mathbf{x}, t)$ is the drift coefficient, $g(t)$ is the diffusion coefficient, and $\mathbf{w}$ is standard Brownian motion (Wiener process).

### Score-Based Generative Models (Diffusion Models)
- **Forward SDE** (Noising): Gradually perturbs data to noise:
  $$d\mathbf{x} = f(t)\mathbf{x} dt + g(t) d\mathbf{w}$$
- **Reverse SDE** (Denoising): Generates data from noise. Crucially, it depends on the **score function** $\nabla_\mathbf{x} \log p_t(\mathbf{x})$:
  $$d\mathbf{x} = \left[ f(t)\mathbf{x} - g(t)^2 \nabla_\mathbf{x} \log p_t(\mathbf{x}) \right] dt + g(t) d\bar{\mathbf{w}}$$
  where $\bar{\mathbf{w}}$ is Brownian motion backward in time, and a neural network (the "score matcher") is trained to predict $\nabla_\mathbf{x} \log p_t(\mathbf{x})$.

In [ ]:
# Simulate simple Ornstein-Uhlenbeck SDE (drift pulls back to zero)
# dx = -theta * x * dt + sigma * dw
N = 1000
t_end = 10.0
dt = t_end / N
t = np.linspace(0, t_end, N)
x = np.zeros(N)
x[0] = 5.0  # start far away

theta = 1.0
sigma = 0.5

for i in range(1, N):
    dw = np.random.normal(0, np.sqrt(dt))
    x[i] = x[i-1] - theta * x[i-1] * dt + sigma * dw

plt.figure(figsize=(8, 4))
plt.plot(t, x, 'b-')
plt.title("Ornstein-Uhlenbeck SDE Simulation")
plt.xlabel("Time t")
plt.ylabel("State x(t)")
plt.grid(True)
plt.show()

## 6. Partial Differential Equations (PDEs) & PINNs

A **Partial Differential Equation (PDE)** contains derivatives with respect to multiple independent variables (e.g. time and space), such as the Heat Equation:
$$\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}$$

**Physics-Informed Neural Networks (PINNs)** solve PDEs by minimizing a loss function that penalizes residuals of the PDE itself at collocation points:
$$\mathcal{L} = \mathcal{L}_{data} + \mathcal{L}_{boundary} + \mathcal{L}_{physics}$$
where $\mathcal{L}_{physics}$ is evaluated using automatic differentiation to compute the PDE terms.